In [ ]:
#From Accuracy to Readiness: Metrics and Benchmarks for Human-AI import pandas as pd
import numpy as np

def generate_mock_data():
    """
    Generates a DataFrame simulating the interaction traces
    required by the paper's evaluation framework.
    
    Variables:
    - y: Ground truth (The correct answer)
    - h0: Initial human decision (before seeing the AI)
    - a: AI prediction or recommendation
    - h1: Final team decision (after seeing the AI)
    - c: User reported confidence (0.0 to 1.0)
    """
    data = {
        'case_id': range(1, 11),
        'y':  [1, 0, 1, 1, 0, 1, 0, 0, 1, 0], # Correct answers
        'h0': [1, 1, 0, 1, 0, 0, 0, 1, 1, 1], # Human makes some mistakes
        'a':  [1, 0, 1, 0, 0, 0, 1, 0, 1, 1], # AI also has some failures
        'h1': [1, 0, 1, 0, 0, 0, 1, 0, 1, 1], # Final team decisions
        'c':  [0.9, 0.8, 0.7, 0.6, 0.9, 0.5, 0.8, 0.7, 0.9, 0.6]
    }
    return pd.DataFrame(data)

def calculate_outcome_metrics(df):
    """Calculates Outcome metrics and Avoidable Error (Regret)."""
    print("--- 1. OUTCOME METRICS ---")
    
    acc_h0 = (df['h0'] == df['y']).mean()
    acc_ai = (df['a'] == df['y']).mean()
    acc_team = (df['h1'] == df['y']).mean()
    
    print(f"Human-Only Accuracy (h0): {acc_h0:.2f}")
    print(f"AI-Only Accuracy (a):     {acc_ai:.2f}")
    print(f"Team Accuracy (h1):       {acc_team:.2f}")
    print(f"Team Gain (vs Human):     {acc_team - acc_h0:.2f}")
    
    # Oracle: 1 if EITHER (initial human or AI) had the right answer
    df['oracle'] = (df['h0'] == df['y']) | (df['a'] == df['y'])
    oracle_acc = df['oracle'].mean()
    
    # Regret_best: Failed even though at least one agent had the correct answer
    regret = (df['oracle'].astype(int) - (df['h1'] == df['y']).astype(int)).mean()
    
    print(f"Oracle Accuracy (Ideal):  {oracle_acc:.2f}")
    print(f"Regret (Avoidable Error): {regret:.2f}\n")

def calculate_reliance_metrics(df):
    """Calculates Reliance metrics (Confidence calibration)."""
    print("--- 2. RELIANCE METRICS ---")
    
    # Pr(h1 = a | a = y): Accept when correct
    correct_ai_mask = df['a'] == df['y']
    accept_on_correct = (df[correct_ai_mask]['h1'] == df[correct_ai_mask]['a']).mean()
    
    # Pr(h1 = a | a != y): Accept when incorrect (Over-reliance)
    wrong_ai_mask = df['a'] != df['y']
    accept_on_wrong = (df[wrong_ai_mask]['h1'] == df[wrong_ai_mask]['a']).mean()
    
    # Reliance Slope
    reliance_slope = accept_on_correct - accept_on_wrong
    
    print(f"Accept-on-Correct (Approp.):  {accept_on_correct:.2f}")
    print(f"Accept-on-Wrong (Danger):     {accept_on_wrong:.2f}")
    print(f"Reliance Slope (Calibration): {reliance_slope:.2f}")
    
    # Decision flips behaviors
    changed = (df['h1'] != df['h0']).mean()
    changed_to_wrong = ((df['h1'] != df['h0']) & (df['h0'] == df['y']) & (df['h1'] != df['y'])).mean()
    
    print(f"Decision Flip Rate:           {changed:.2f}")
    print(f"Changed-to-Wrong (Auto Bias): {changed_to_wrong:.2f}\n")

def calculate_safety_metrics(df):
    """Calculates Safety metrics separating AI help from AI harm."""
    print("--- 3. SAFETY METRICS (HELP-HARM) ---")
    
    # AI-Help: Human was originally wrong, saw AI, and corrected it
    ai_help = ((df['h0'] != df['y']) & (df['h1'] == df['y'])).mean()
    
    # AI-Harm: Human was originally right, saw AI, and changed to wrong
    ai_harm = ((df['h0'] == df['y']) & (df['h1'] != df['y'])).mean()
    
    # Missed-Help: Human was wrong, AI was right, but human ignored the AI
    missed_help = ((df['h0'] != df['y']) & (df['a'] == df['y']) & (df['h1'] != df['a'])).mean()
    
    print(f"AI-Help (Effective Help):     {ai_help:.2f}")
    print(f"AI-Harm (Induced Harm):       {ai_harm:.2f}")
    print(f"Missed-Help (Under-reliance): {missed_help:.2f}\n")

def calculate_learning_metrics(df):
    """Calculates learning metrics, such as the calibration gap."""
    print("--- 4. LEARNING METRICS ---")
    
    # Calibration gap: Difference between reported confidence and actual correctness
    is_correct = (df['h1'] == df['y']).astype(int)
    calibration_gap = (df['c'] - is_correct).abs().mean()
    
    print(f"Mean Calibration Gap:         {calibration_gap:.2f}")
    print("  (Close to 0 means excellent calibration between confidence and reality)\n")

if __name__ == "__main__":
    print("Human-AI System Evaluation: Running metrics from logs...\n")
    df_logs = generate_mock_data()
    
    calculate_outcome_metrics(df_logs)
    calculate_reliance_metrics(df_logs)
    calculate_safety_metrics(df_logs)
    calculate_learning_metrics(df_logs)